# OpenWakeWord: Custom Human Voice Training
This notebook is optimized for training a wake word model primarily on your **real voice recordings**.
It still generates a small amount of synthetic data to create "hard negatives" (words that sound like your wake word but aren't) to prevent false activations, but the vast majority of the training will focus on your actual voice!

### Step 1: Install Dependencies
Run this cell to install OpenWakeWord and download the required background noise datasets.

In [ ]:
!git clone https://github.com/dscripka/openWakeWord.git
!git clone https://github.com/dscripka/piper-sample-generator
!wget -O piper-sample-generator/models/en_US-libritts_r-medium.pt 'https://github.com/rhasspy/piper-sample-generator/releases/download/v2.0.0/en_US-libritts_r-medium.pt'
!sed -i 's/torch.load(model_path)/torch.load(model_path, weights_only=False)/g' /content/openWakeWord/piper-sample-generator/generate_samples.py
!sed -i 's/torch.load(checkpoint_path, map_location=device)/torch.load(checkpoint_path, map_location=device, weights_only=False)/g' /usr/local/lib/python3.12/dist-packages/dp/model/model.py
!pip install webrtcvad
!pip install mutagen==1.47.0
!pip install torchinfo==1.8.0
!pip install torchmetrics==1.2.0
!pip install speechbrain==0.5.14
!pip install audiomentations==0.33.0
!pip install torch-audiomentations==0.12.0
!pip install acoustics==0.2.6
!pip install pronouncing==0.2.0
!pip install datasets==2.14.6
!pip install deep-phonemizer==0.0.19
!pip install espeak-phonemizer
%cd openWakeWord
!pip install -e .
!sed -i 's/convert_onnx_to_tflite/#convert_onnx_to_tflite/g' openwakeword/train.py
!pip install pronouncing
!pip install audiomentations
!pip install torch-audiomentations
!pip install speechbrain
!pip install mutagen
!pip install acoustics
!pip install torchinfo
!pip install torchmetrics
import os
os.makedirs('./openwakeword/resources/models', exist_ok=True)
!wget https://github.com/dscripka/openWakeWord/releases/download/v0.5.1/embedding_model.onnx -O ./openwakeword/resources/models/embedding_model.onnx
!wget https://github.com/dscripka/openWakeWord/releases/download/v0.5.1/melspectrogram.onnx -O ./openwakeword/resources/models/melspectrogram.onnx
!wget https://huggingface.co/datasets/davidscripka/openwakeword_features/resolve/main/openwakeword_features_ACAV100M_2000_hrs_16bit.npy
!wget https://huggingface.co/datasets/davidscripka/openwakeword_features/resolve/main/validation_set_features.npy

### Step 2: Upload Your Voice Clips
1. Record 50-100 clips of yourself saying your wake word (16kHz, 16-bit, Mono `.wav` format).
2. Put them in a `.zip` file named `my_positive_clips.zip`.
3. Upload that `.zip` file to the Colab sidebar.
4. Run the cell below to extract them!

In [ ]:
!unzip /content/my_positive_clips.zip -d /content/
print("Voice clips extracted to /content/my_positive_clips/")

### Step 3: Configure Your Wake Word
Change the `target_phrase` below to the word(s) you recorded yourself saying.

In [ ]:
import yaml
import os

# Load the default config from openWakeWord to ensure no missing keys!
with open('/content/openWakeWord/examples/custom_model.yml', 'r') as f:
    config = yaml.load(f.read(), yaml.Loader)

# Override with our custom settings
config["target_phrase"] = ["ultron"] # Change this to your wake word
config["model_name"] = "custom_voice_model"
config["piper_sample_generator_path"] = "/content/openWakeWord/piper-sample-generator"
config["n_samples"] = 100 # Keep low so real voice dominates
config["n_samples_val"] = 100
config["steps"] = 10000
config["target_accuracy"] = 0.6
config["target_recall"] = 0.25
config["background_paths"] = ['./audioset_16k', './fma']
config["false_positive_validation_data_path"] = "validation_set_features.npy"
config["feature_data_files"] = {"ACAV100M_sample": "openwakeword_features_ACAV100M_2000_hrs_16bit.npy"}

with open('my_model.yaml', 'w') as file:
    yaml.dump(config, file)
print("Configuration saved successfully with all default keys intact!")


### Step 4: Generate Synthetic Negatives & Phonetic Variations
This runs quickly because we lowered the `n_samples`. It generates words that sound similar to your wake word to teach the model what *not* to trigger on.

In [ ]:
import os
import scipy.io.wavfile
import numpy as np
from tqdm import tqdm
import datasets

output_dir = "./mit_rirs"
if not os.path.exists(output_dir):
    os.mkdir(output_dir)
rir_dataset = datasets.load_dataset("davidscripka/MIT_environmental_impulse_responses", split="train", streaming=True)

print('Downloading Room Impulse Responses (for augmentation)...')
for row in tqdm(rir_dataset):
    name = row['audio']['path'].split('/')[-1]
    scipy.io.wavfile.write(os.path.join(output_dir, name), 16000, (row['audio']['array']*32767).astype(np.int16))
print('Finished downloading RIRs!')


In [ ]:
import os
import scipy.io.wavfile
import numpy as np
from tqdm import tqdm
import datasets
from pathlib import Path

print('Downloading Audioset background noise...')
if not os.path.exists("audioset"):
    os.mkdir("audioset")

fname = "bal_train09.tar"
out_dir = f"audioset/{fname}"
link = "https://huggingface.co/datasets/agkphysics/AudioSet/resolve/main/data/" + fname
!wget -O {out_dir} {link}
!cd audioset && tar -xvf bal_train09.tar

output_dir = "./audioset_16k"
if not os.path.exists(output_dir):
    os.mkdir(output_dir)

audioset_dataset = datasets.Dataset.from_dict({"audio": [str(i) for i in Path("audioset/audio").glob("**/*.flac")]})
audioset_dataset = audioset_dataset.cast_column("audio", datasets.Audio(sampling_rate=16000))
for row in tqdm(audioset_dataset):
    name = row['audio']['path'].split('/')[-1].replace(".flac", ".wav")
    scipy.io.wavfile.write(os.path.join(output_dir, name), 16000, (row['audio']['array']*32767).astype(np.int16))

print('Downloading FMA background noise...')
output_dir = "./fma"
if not os.path.exists(output_dir):
    os.mkdir(output_dir)
fma_dataset = datasets.load_dataset("rudraml/fma", name="small", split="train", streaming=True)
fma_dataset = iter(fma_dataset.cast_column("audio", datasets.Audio(sampling_rate=16000)))

n_hours = 1
for i in tqdm(range(n_hours*3600//30)):
    row = next(fma_dataset)
    name = row['audio']['path'].split('/')[-1].replace(".mp3", ".wav")
    scipy.io.wavfile.write(os.path.join(output_dir, name), 16000, (row['audio']['array']*32767).astype(np.int16))
    i += 1
    if i == n_hours*3600//30:
        break
print('Finished downloading background noise!')


In [ ]:
import sys
!{sys.executable} openwakeword/train.py --training_config my_model.yaml --generate_clips
!{sys.executable} openwakeword/train.py --training_config my_model.yaml --augment_clips

### Step 5: Inject Your Real Human Voice Data
This cell converts your `.wav` files into OpenWakeWord features and seamlessly merges them into the training dataset.

In [ ]:
import os
import glob
import numpy as np
from openwakeword.utils import AudioFeatures

custom_clips_dir = "/content/my_positive_clips"
if os.path.exists(custom_clips_dir):
    positive_clips = glob.glob(os.path.join(custom_clips_dir, "*.wav"))
    if len(positive_clips) > 0:
        print(f"Found {len(positive_clips)} custom positive clips. Extracting features...")
        F = AudioFeatures(device="cpu")
        custom_features = F.embed_clips(positive_clips, batch_size=16)

        if isinstance(custom_features, dict):
            custom_features = list(custom_features.values())[0]
        if isinstance(custom_features, list):
            custom_features = np.vstack(custom_features)

        model_name = config["model_name"]
        output_dir = os.path.join(config.get("output_dir", "my_custom_model"), model_name)
        train_path = os.path.join(output_dir, "positive_features_train.npy")
        val_path = os.path.join(output_dir, "positive_features_val.npy")

        if os.path.exists(train_path):
            existing_train = np.load(train_path)
            # We repeat the custom human features multiple times so the model focuses heavily on them.
            # MULTIPLIER: Set this to 4 (or higher) to repeat your 26 recordings multiple times.
            weight_multiplier = 4
            weighted_custom = np.tile(custom_features, (weight_multiplier, 1))
            new_train = np.vstack([existing_train, weighted_custom])
            np.save(train_path, new_train)
            print(f"Appended weighted real clips to training features. New total: {new_train.shape[0]}")

        if os.path.exists(val_path):
            existing_val = np.load(val_path)
            val_features = custom_features[:max(1, len(custom_features)//5)]
            new_val = np.vstack([existing_val, val_features])
            np.save(val_path, new_val)
            print(f"Appended real clips to validation features. New total: {new_val.shape[0]}")
    else:
        print(f"Directory {custom_clips_dir} exists, but no .wav files found.")
else:
    print(f"No custom real clips found at {custom_clips_dir}. Skipping...")

### Step 6: Train the Model!
Train the final neural network. Once complete, download the `.onnx` file from `/content/openWakeWord/my_custom_model/custom_voice_model/`.

In [ ]:
import sys
!{sys.executable} openwakeword/train.py --training_config my_model.yaml --train_model